# INP-Former — MVTec AD 2 — Single-Class — ONE CATEGORY / RUN

Bản này được tổ chức lại để phù hợp giới hạn GPU Colab:

- **Mỗi lần chạy chỉ chọn đúng 1 trong 8 category.**
- Không có cell nào tự động train cả 8 category.
- Các category chưa dùng được để dưới dạng **comment** trong cell cấu hình, giống cách tổ chức notebook train mẫu.
- Dataset cũng chỉ extract category đang chọn để giảm dung lượng local.
- Checkpoint, raw map, metric và private outputs được lưu **riêng theo category trên Google Drive**, nên có thể chạy nhiều phiên Colab khác nhau và tích lũy kết quả.
- Khi đã chạy đủ 8 category, cell tổng hợp cuối cùng tự tạo `week04_primary.csv` và macro mean.

## Protocol INP-Former khóa

- Official repo: `https://github.com/luow23/INP-Former`
- Commit: `17d265381d9b323a2ef6e05aab0665a85edebe84`
- Setting: **single-class**
- Backbone: `dinov2reg_vit_base_14`
- Official preprocessing: `Resize 448 -> CenterCrop 392 -> ImageNet Normalize`
- `INP_num = 6`
- `epochs = 200`
- Train chỉ dùng `train/good`
- Public evaluation: `test_public`
- Primary metric bổ sung: **AU-PRO@FPR=0.05**
- AU-PRO dùng cùng logic region-equal-weight / threshold sweep / interpolation tại FPR=0.05 như notebook SuperADD mẫu.

## 1. Mount Drive + CHỌN 1 CATEGORY

**Chỉ sửa `CATEGORY` ở cell này.**

Ví dụ muốn chạy `sheet_metal` thì để:
```python
CATEGORY = "sheet_metal"
```

Các category còn lại giữ comment. Khi muốn chuyển category, đổi đúng một dòng rồi **Restart runtime + Run all**.

In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path, PurePosixPath
import os
import sys
import json
import shutil
import subprocess
import hashlib
import tarfile

# ============================================================
# PROJECT PATH
# ============================================================

PROJECT_DRIVE = Path(
    "/content/drive/MyDrive/[Q3-4] 2026/[S7] Computer Vision/CV-Nhóm 9"
)

ARCHIVE_DIR = PROJECT_DRIVE / "data"
DRIVE_OUT = PROJECT_DRIVE / "inp_former_results_single_class"

# ============================================================
# CHỌN ĐÚNG 1 CATEGORY / 1 LẦN CHẠY
# ============================================================

# CATEGORY = "vial"

CATEGORY = "can"
# CATEGORY = "fabric"
# CATEGORY = "fruit_jelly"
# CATEGORY = "rice"
# CATEGORY = "sheet_metal"
# CATEGORY = "wallplugs"
# CATEGORY = "walnuts"

ALL_CATEGORIES = [
    "can",
    "fabric",
    "fruit_jelly",
    "rice",
    "sheet_metal",
    "vial",
    "wallplugs",
    "walnuts",
]

assert CATEGORY in ALL_CATEGORIES, CATEGORY

# Chỉ 1 phần tử. Các cell phía sau đều dùng biến này.
CATEGORIES = [CATEGORY]

# ============================================================
# REPO / DATA LOCAL
# ============================================================

REPO = Path("/content/INP-Former")

# Native MVTec AD 2 local — chỉ category đang chọn được extract.
MAD2_NATIVE_ROOT = REPO / "data" / "mvtec_ad_2"

# Staging schema tương thích official MVTecDataset.
INP_DATA_ROOT = REPO / "data" / "mvtec_ad_2_inp"

COMMIT = "17d265381d9b323a2ef6e05aab0665a85edebe84"

# ============================================================
# OFFICIAL SINGLE-CLASS CONFIG
# ============================================================

INPUT_SIZE = 448
CROP_SIZE = 392
INP_NUM = 6
EPOCHS = 200
TRAIN_BATCH = 16
NUM_WORKERS = 2
SEED = 1

# Nếu L4 OOM thì chỉ giảm batch:
# TRAIN_BATCH = 8

# ============================================================
# OPTIONAL FLAGS
# ============================================================

FORCE_EXTRACT = False
FORCE_TRAIN = False
FORCE_PUBLIC_INFERENCE = False

# Private inference tốn thêm GPU/storage.
# Chỉ bật khi public + metric của category đã PASS.
RUN_PRIVATE_INFERENCE = False

# Failure gallery là optional.
RUN_FAILURE_ANALYSIS = False

assert ARCHIVE_DIR.is_dir(), ARCHIVE_DIR
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("SELECTED CATEGORY :", CATEGORY)
print("ARCHIVE_DIR       :", ARCHIVE_DIR)
print("DRIVE_OUT         :", DRIVE_OUT)
print("=" * 70)

Mounted at /content/drive
SELECTED CATEGORY : can
ARCHIVE_DIR       : /content/drive/MyDrive/[Q3-4] 2026/[S7] Computer Vision/CV-Nhóm 9/data
DRIVE_OUT         : /content/drive/MyDrive/[Q3-4] 2026/[S7] Computer Vision/CV-Nhóm 9/inp_former_results_single_class


## 2. GPU audit + clone đúng official INP-Former

In [2]:
def run(cmd, cwd=None, env=None):
    cmd = [str(x) for x in cmd]
    print("$", " ".join(cmd))
    return subprocess.run(cmd, cwd=cwd, env=env, check=True)

run(["nvidia-smi"])

if REPO.exists():
    shutil.rmtree(REPO)

run([
    "git",
    "clone",
    "https://github.com/luow23/INP-Former.git",
    REPO,
])

run([
    "git",
    "checkout",
    "--detach",
    COMMIT,
], cwd=REPO)

actual_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO,
    text=True,
).strip()

assert actual_commit == COMMIT, (actual_commit, COMMIT)

# Không pip install nguyên requirements.txt vì repo pin nhiều package rất cũ
# so với Colab hiện tại. Giữ Torch/CUDA của Colab.
run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "timm==0.9.12",
    "kornia==0.7.3",
    "ADEval==1.1.0",
    "tabulate>=0.9",
    "tifffile>=2023.7",
    "scikit-image>=0.21",
    "scikit-learn>=1.3",
    "opencv-python-headless>=4.8",
])

dirty = subprocess.check_output(
    ["git", "status", "--porcelain"],
    cwd=REPO,
    text=True,
).strip()

assert dirty == "", f"Official source bị thay đổi:\n{dirty}"

print("REPO PASS")
print("Commit:", actual_commit)

$ nvidia-smi
$ git clone https://github.com/luow23/INP-Former.git /content/INP-Former
$ git checkout --detach 17d265381d9b323a2ef6e05aab0665a85edebe84
$ /usr/bin/python3 -m pip install -q timm==0.9.12 kornia==0.7.3 ADEval==1.1.0 tabulate>=0.9 tifffile>=2023.7 scikit-image>=0.21 scikit-learn>=1.3 opencv-python-headless>=4.8
REPO PASS
Commit: 17d265381d9b323a2ef6e05aab0665a85edebe84


## 3. Extract CHỈ category đang chọn

Notebook tìm:
```text
data/<CATEGORY>.tar.gz
```

Ví dụ khi `CATEGORY="vial"` thì chỉ giải nén `data/vial.tar.gz`.

Các archive của 7 category còn lại **không được giải nén** trong phiên này.

In [3]:
SPLITS_NEEDED = {
    "train",
    "test_public",
    "test_private",
    "test_private_mixed",
}

archive = ARCHIVE_DIR / f"{CATEGORY}.tar.gz"

assert archive.is_file(), (
    f"Không tìm thấy archive cho category đang chọn:\n{archive}"
)

category_native_root = MAD2_NATIVE_ROOT / CATEGORY

def split_has_files(root: Path, split: str):
    split_root = root / split
    return split_root.exists() and any(split_root.rglob("*.png"))

def extract_selected_splits(
    archive_path: Path,
    destination: Path,
    splits=SPLITS_NEEDED,
    force=False,
):
    destination.mkdir(parents=True, exist_ok=True)

    if not force:
        complete = all(
            split_has_files(destination, split)
            for split in splits
        )
        if complete:
            print("[SKIP EXTRACT] Dataset đã có đủ split:", destination)
            return

    for split in splits:
        shutil.rmtree(destination / split, ignore_errors=True)

    copied = {split: 0 for split in splits}

    with tarfile.open(archive_path, "r:gz") as tf:
        for member in tf.getmembers():
            parts = PurePosixPath(member.name).parts

            positions = [
                i
                for i, part in enumerate(parts)
                if part in splits
            ]

            if not positions:
                continue

            i = positions[0]
            split = parts[i]
            rel_parts = parts[i:]

            if any(part in ("", ".", "..") for part in rel_parts):
                raise RuntimeError(f"Unsafe archive path: {member.name}")

            target = destination.joinpath(*rel_parts)

            if member.isdir():
                target.mkdir(parents=True, exist_ok=True)
                continue

            if not member.isfile():
                continue

            target.parent.mkdir(parents=True, exist_ok=True)

            src = tf.extractfile(member)
            if src is None:
                raise RuntimeError(member.name)

            with src, open(target, "wb") as dst:
                shutil.copyfileobj(
                    src,
                    dst,
                    length=8 * 1024 * 1024,
                )

            copied[split] += 1

    for split in splits:
        assert split_has_files(destination, split), (
            f"Extract lỗi: {CATEGORY}/{split}"
        )

    print("Extract counts:", copied)

extract_selected_splits(
    archive,
    category_native_root,
    force=FORCE_EXTRACT,
)

# ============================================================
# DATASET AUDIT — CATEGORY ĐANG CHỌN
# ============================================================

root = category_native_root

checks = {
    "train/good":
        len(list((root / "train" / "good").glob("*.png"))),

    "test_public/good":
        len(list((root / "test_public" / "good").glob("*.png"))),

    "test_public/bad":
        len(list((root / "test_public" / "bad").glob("*.png"))),

    "test_public/ground_truth/bad":
        len(list(
            (root / "test_public" / "ground_truth" / "bad")
            .glob("*_mask.png")
        )),

    "test_private":
        len(list((root / "test_private").glob("*.png"))),

    "test_private_mixed":
        len(list((root / "test_private_mixed").glob("*.png"))),
}

for key, value in checks.items():
    print(f"{key:35s}: {value}")
    assert value > 0, f"{CATEGORY}: thiếu {key}"

print("DATASET AUDIT PASS:", CATEGORY)

Extract counts: {'train': 412, 'test_private_mixed': 321, 'test_public': 252, 'test_private': 321}
train/good                         : 412
test_public/good                   : 72
test_public/bad                    : 90
test_public/ground_truth/bad       : 90
test_private                       : 321
test_private_mixed                 : 321
DATASET AUDIT PASS: can


## 4. Tạo staging schema đúng chuẩn INP-Former — chỉ selected category

Official `MVTecDataset` cần:

```text
<CATEGORY>/
├── train/good/
├── test/good/
├── test/bad/
└── ground_truth/bad/
```

MVTec AD 2 public lại nằm trong `test_public`, vì vậy notebook tạo symlink local.

In [4]:
def link_dir(src: Path, dst: Path):
    if dst.is_symlink() or dst.is_file():
        dst.unlink()
    elif dst.exists():
        shutil.rmtree(dst)

    dst.parent.mkdir(parents=True, exist_ok=True)

    os.symlink(
        src,
        dst,
        target_is_directory=True,
    )

staged_root = INP_DATA_ROOT / CATEGORY

if staged_root.exists() or staged_root.is_symlink():
    shutil.rmtree(staged_root, ignore_errors=True)

native = MAD2_NATIVE_ROOT / CATEGORY

link_dir(
    native / "train" / "good",
    staged_root / "train" / "good",
)

link_dir(
    native / "test_public" / "good",
    staged_root / "test" / "good",
)

link_dir(
    native / "test_public" / "bad",
    staged_root / "test" / "bad",
)

link_dir(
    native / "test_public" / "ground_truth" / "bad",
    staged_root / "ground_truth" / "bad",
)

assert list((staged_root / "train" / "good").glob("*.png"))
assert list((staged_root / "test" / "good").glob("*.png"))
assert list((staged_root / "test" / "bad").glob("*.png"))
assert list((staged_root / "ground_truth" / "bad").glob("*_mask.png"))

print("STAGING PASS:", staged_root)

STAGING PASS: /content/INP-Former/data/mvtec_ad_2_inp/can


## 5. Import official modules + model builder

In [5]:
os.chdir(REPO)
sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import cv2
import tifffile

from functools import partial
from tqdm.auto import tqdm
from PIL import Image

from torch.nn.init import trunc_normal_
from torch.utils.data import DataLoader, Dataset
from torchvision.datasets import ImageFolder

from optimizers import StableAdamW

from utils import (
    WarmCosineScheduler,
    global_cosine_hm_adaptive,
    setup_seed,
    cal_anomaly_maps,
    get_gaussian_kernel,
)

from dataset import (
    MVTecDataset,
    get_data_transforms,
)

from models import vit_encoder
from models.uad import INP_Former

from models.vision_transformer import (
    Mlp,
    Aggregation_Block,
    Prototype_Block,
)

device = torch.device("cuda:0")

assert torch.cuda.is_available()

setup_seed(SEED)

data_transform, gt_transform = get_data_transforms(
    INPUT_SIZE,
    CROP_SIZE,
)

def build_model():
    target_layers = [
        2, 3, 4, 5,
        6, 7, 8, 9,
    ]

    fuse_layer_encoder = [
        [0, 1, 2, 3],
        [4, 5, 6, 7],
    ]

    fuse_layer_decoder = [
        [0, 1, 2, 3],
        [4, 5, 6, 7],
    ]

    encoder = vit_encoder.load(
        "dinov2reg_vit_base_14"
    )

    embed_dim = 768
    num_heads = 12

    bottleneck = nn.ModuleList([
        Mlp(
            embed_dim,
            embed_dim * 4,
            embed_dim,
            drop=0.0,
        )
    ])

    inp = nn.ParameterList([
        nn.Parameter(
            torch.randn(
                INP_NUM,
                embed_dim,
            )
        )
    ])

    extractor = nn.ModuleList([
        Aggregation_Block(
            dim=embed_dim,
            num_heads=num_heads,
            mlp_ratio=4.0,
            qkv_bias=True,
            norm_layer=partial(
                nn.LayerNorm,
                eps=1e-8,
            ),
        )
    ])

    decoder = nn.ModuleList([
        Prototype_Block(
            dim=embed_dim,
            num_heads=num_heads,
            mlp_ratio=4.0,
            qkv_bias=True,
            norm_layer=partial(
                nn.LayerNorm,
                eps=1e-8,
            ),
        )
        for _ in range(8)
    ])

    model = INP_Former(
        encoder=encoder,
        bottleneck=bottleneck,
        aggregation=extractor,
        decoder=decoder,
        target_layers=target_layers,
        remove_class_token=True,
        fuse_layer_encoder=fuse_layer_encoder,
        fuse_layer_decoder=fuse_layer_decoder,
        prototype_token=inp,
    ).to(device)

    trainable = nn.ModuleList([
        bottleneck,
        decoder,
        extractor,
        inp,
    ])

    # Official single-class initialization.
    for module in trainable.modules():
        if isinstance(module, nn.Linear):
            trunc_normal_(
                module.weight,
                std=0.01,
                a=-0.03,
                b=0.03,
            )

            if module.bias is not None:
                nn.init.constant_(
                    module.bias,
                    0,
                )

        elif isinstance(module, nn.LayerNorm):
            nn.init.constant_(
                module.bias,
                0,
            )

            nn.init.constant_(
                module.weight,
                1.0,
            )

    return model, trainable

print("GPU       :", torch.cuda.get_device_name(0))
print("Torch     :", torch.__version__)
print("CUDA      :", torch.version.cuda)
print("Category  :", CATEGORY)
print("Preprocess: Resize 448 -> CenterCrop 392")

/usr/local/lib/python3.13/dist-packages/kornia/feature/lightglue.py:44: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)


GPU       : Tesla T4
Torch     : 2.11.0+cu128
CUDA      : 12.8
Category  : can
Preprocess: Resize 448 -> CenterCrop 392


## 6. TRAIN selected category

Cell này chỉ train:
```python
CATEGORY
```

Không có vòng `for` chạy 8 category.

Checkpoint được lưu trên Drive nên sau khi xong một category, bạn đổi `CATEGORY` và chạy phiên mới.

In [ ]:
SAVE_NAME = (
    "INP-Former-Single-Class"
    "_dataset=MVTec-AD2"
    "_Encoder=dinov2reg_vit_base_14"
    f"_Resize={INPUT_SIZE}"
    f"_Crop={CROP_SIZE}"
    f"_INP_num={INP_NUM}"
)

CHECKPOINT_ROOT = (
    DRIVE_OUT
    / "saved_results"
    / SAVE_NAME
)

CHECKPOINT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

SAFE_NUM_WORKERS = 0

# Lưu sau MỖI epoch.
SAVE_EVERY = 1


def sha256_file(path, chunk=8 * 1024 * 1024):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for data in iter(
            lambda: f.read(chunk),
            b"",
        ):
            h.update(data)

    return h.hexdigest()


# ============================================================
# SAVE ATOMIC
# ============================================================
# Ghi file tạm trước rồi replace.
# Giảm nguy cơ checkpoint hỏng nếu Colab ngắt đúng lúc torch.save.
# ============================================================

def atomic_torch_save(state, path):

    path = Path(path)

    tmp_path = path.with_name(
        path.name + ".tmp"
    )

    torch.save(
        state,
        tmp_path,
    )

    os.replace(
        tmp_path,
        path,
    )


# ============================================================
# TRAIN
# ============================================================

def train_selected_category(
    category,
    force=False,
    save_every=SAVE_EVERY,
):

    out_dir = (
        CHECKPOINT_ROOT
        / category
    )

    out_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    # ========================================================
    # CHỈ DÙNG 2 CHECKPOINT CHÍNH
    # ========================================================

    last_ckpt = (
        out_dir
        / "last.pth"
    )

    best_ckpt = (
        out_dir
        / "best.pth"
    )

    history_path = (
        out_dir
        / "train_history.csv"
    )


    # ========================================================
    # REPRODUCIBILITY
    # ========================================================

    setup_seed(SEED)


    # ========================================================
    # DATASET
    # ========================================================

    train_data = ImageFolder(
        root=str(
            INP_DATA_ROOT
            / category
            / "train"
        ),
        transform=data_transform,
    )


    train_loader = DataLoader(
        train_data,

        batch_size=TRAIN_BATCH,

        shuffle=True,

        num_workers=SAFE_NUM_WORKERS,

        pin_memory=True,

        drop_last=True,
    )


    # ========================================================
    # MODEL
    # ========================================================

    model, trainable = build_model()


    # ========================================================
    # OPTIMIZER
    # ========================================================

    optimizer = StableAdamW(
        [
            {
                "params":
                    trainable.parameters()
            }
        ],

        lr=1e-3,

        betas=(
            0.9,
            0.999,
        ),

        weight_decay=1e-4,

        amsgrad=True,

        eps=1e-10,
    )


    # ========================================================
    # SCHEDULER
    # ========================================================

    scheduler = WarmCosineScheduler(
        optimizer,

        base_value=1e-3,

        final_value=1e-4,

        total_iters=(
            EPOCHS
            * len(train_loader)
        ),

        warmup_iters=100,
    )


    # ========================================================
    # INITIAL STATE
    # ========================================================

    start_epoch = 0

    history = []

    best_loss = float("inf")

    best_epoch = 0


    # ========================================================
    # LOAD HISTORY
    # ========================================================

    if (
        history_path.exists()
        and not force
    ):

        try:

            old_history = pd.read_csv(
                history_path
            )

            history = (
                old_history
                .to_dict(
                    orient="records"
                )
            )

        except Exception as e:

            print(
                "[WARNING] Cannot load history:",
                e,
            )


    # ========================================================
    # RESUME FROM last.pth
    # ========================================================

    if (
        last_ckpt.exists()
        and not force
    ):

        print(
            "[LOAD RESUME]",
            last_ckpt,
        )


        # ----------------------------------------------------
        # QUAN TRỌNG:
        #
        # PyTorch >= 2.6 mặc định weights_only=True.
        #
        # last.pth chứa optimizer + scheduler + numpy objects,
        # vì vậy cần weights_only=False.
        #
        # Chỉ sử dụng vì đây là checkpoint DO CHÍNH TA TẠO.
        # ----------------------------------------------------

        state = torch.load(
            last_ckpt,

            map_location="cpu",

            weights_only=False,
        )


        # ----------------------------------------------------
        # MODEL
        # ----------------------------------------------------

        model.load_state_dict(
            state["model"],
            strict=True,
        )


        # ----------------------------------------------------
        # OPTIMIZER
        # ----------------------------------------------------

        optimizer.load_state_dict(
            state["optimizer"]
        )


        # ----------------------------------------------------
        # SCHEDULER
        # ----------------------------------------------------

        scheduler.load_state_dict(
            state["scheduler"]
        )


        # ----------------------------------------------------
        # EPOCH
        # ----------------------------------------------------

        start_epoch = (
            int(
                state["epoch"]
            )
            + 1
        )


        # ----------------------------------------------------
        # BEST STATE
        # Checkpoint mới sẽ có 2 field này.
        # ----------------------------------------------------

        if "best_loss" in state:

            best_loss = float(
                state["best_loss"]
            )

            best_epoch = int(
                state.get(
                    "best_epoch",
                    0,
                )
            )


        # ----------------------------------------------------
        # MIGRATION:
        # last.pth cũ của bạn epoch 26 chưa có best_loss.
        # ----------------------------------------------------

        else:

            current_epoch = (
                int(state["epoch"])
                + 1
            )

            current_loss = None

            for row in history:

                if (
                    int(row["epoch"])
                    == current_epoch
                ):

                    current_loss = float(
                        row["loss"]
                    )

                    break


            if current_loss is None:

                print(
                    "[WARNING] "
                    "Không tìm thấy loss của checkpoint cũ."
                )

                current_loss = float(
                    "inf"
                )


            best_loss = current_loss

            best_epoch = current_epoch


            # -----------------------------------------------
            # Tạo best.pth ban đầu từ last.pth hiện có.
            #
            # Ví dụ của bạn:
            # epoch 26 -> dùng epoch 26 làm best baseline.
            # -----------------------------------------------

            migrated_best_state = {
                "epoch":
                    state["epoch"],

                "model":
                    state["model"],

                "optimizer":
                    state["optimizer"],

                "scheduler":
                    state["scheduler"],

                "category":
                    category,

                "input_size":
                    INPUT_SIZE,

                "crop_size":
                    CROP_SIZE,

                "inp_num":
                    INP_NUM,

                "repo_commit":
                    COMMIT,

                "train_loss":
                    current_loss,

                "best_loss":
                    best_loss,

                "best_epoch":
                    best_epoch,
            }


            atomic_torch_save(
                migrated_best_state,
                best_ckpt,
            )


            print(
                "[MIGRATE] Created best.pth "
                f"from existing epoch {current_epoch}"
            )


        print(
            f"[RESUME] {category}: "
            f"last completed epoch = {start_epoch}"
        )

        print(
            f"[BEST] epoch={best_epoch}, "
            f"loss={best_loss:.6f}"
        )


    # ========================================================
    # FORCE = TRUE
    # ========================================================

    elif force:

        print(
            "[FORCE TRAIN] Start from epoch 1"
        )


    # ========================================================
    # INFORMATION
    # ========================================================

    print(
        "=" * 70
    )

    print(
        "TRAIN CATEGORY:",
        category,
    )

    print(
        "Train images  :",
        len(train_data),
    )

    print(
        "Epochs        :",
        EPOCHS,
    )

    print(
        "Start epoch   :",
        start_epoch + 1,
    )

    print(
        "Batch         :",
        TRAIN_BATCH,
    )

    print(
        "Batches/epoch :",
        len(train_loader),
    )

    print(
        "Data workers  :",
        SAFE_NUM_WORKERS,
    )

    print(
        "GPU           :",
        torch.cuda.get_device_name(0),
    )

    print(
        "Best loss     :",
        (
            f"{best_loss:.6f}"
            if np.isfinite(best_loss)
            else "None"
        ),
    )

    print(
        "=" * 70
    )


    # ========================================================
    # TRAIN LOOP
    # ========================================================

    for epoch in range(
        start_epoch,
        EPOCHS,
    ):

        model.train()

        losses = []


        pbar = tqdm(
            train_loader,

            desc=(
                f"{category} "
                f"{epoch + 1}/{EPOCHS}"
            ),

            leave=False,
        )


        # ====================================================
        # BATCH LOOP
        # ====================================================

        for image, _ in pbar:

            image = image.to(
                device,
                non_blocking=True,
            )


            # ------------------------------------------------
            # FORWARD
            # ------------------------------------------------

            en, de, g_loss = model(
                image
            )


            reconstruction_loss = (
                global_cosine_hm_adaptive(
                    en,
                    de,
                    y=3,
                )
            )


            loss = (
                reconstruction_loss
                + 0.2
                * g_loss
            )


            # ------------------------------------------------
            # BACKWARD
            # ------------------------------------------------

            optimizer.zero_grad(
                set_to_none=True,
            )


            loss.backward()


            nn.utils.clip_grad_norm_(
                trainable.parameters(),

                max_norm=0.1,
            )


            optimizer.step()

            scheduler.step()


            # ------------------------------------------------
            # LOG
            # ------------------------------------------------

            loss_value = float(
                loss.item()
            )

            losses.append(
                loss_value
            )


            pbar.set_postfix(
                loss=(
                    f"{np.mean(losses):.4f}"
                )
            )


        # ====================================================
        # END EPOCH
        # ====================================================

        mean_loss = float(
            np.mean(losses)
        )


        current_epoch = (
            epoch + 1
        )


        # ====================================================
        # UPDATE HISTORY
        # ====================================================

        history = [
            row
            for row in history
            if int(row["epoch"])
            != current_epoch
        ]


        history.append(
            {
                "epoch":
                    current_epoch,

                "loss":
                    mean_loss,
            }
        )


        history = sorted(
            history,

            key=lambda x: int(
                x["epoch"]
            ),
        )


        pd.DataFrame(
            history
        ).to_csv(
            history_path,

            index=False,
        )


        print(
            f"{category}: "
            f"epoch {current_epoch:03d}/{EPOCHS} "
            f"loss={mean_loss:.6f}"
        )


        # ====================================================
        # KIỂM TRA BEST
        # ====================================================

        is_best = (
            mean_loss
            < best_loss
        )


        if is_best:

            best_loss = (
                mean_loss
            )

            best_epoch = (
                current_epoch
            )


        # ====================================================
        # CHECKPOINT STATE
        # ====================================================

        checkpoint_state = {

            # Python epoch index.
            "epoch":
                epoch,

            "model":
                model.state_dict(),

            "optimizer":
                optimizer.state_dict(),

            "scheduler":
                scheduler.state_dict(),

            "category":
                category,

            "input_size":
                INPUT_SIZE,

            "crop_size":
                CROP_SIZE,

            "inp_num":
                INP_NUM,

            "repo_commit":
                COMMIT,

            "train_loss":
                mean_loss,

            "best_loss":
                best_loss,

            "best_epoch":
                best_epoch,
        }


        # ====================================================
        # 1. SAVE last.pth
        # ====================================================
        #
        # ALWAYS = model sau epoch vừa hoàn thành.
        # ====================================================

        if (
            current_epoch
            % save_every
            == 0
            or current_epoch
            == EPOCHS
        ):

            atomic_torch_save(
                checkpoint_state,

                last_ckpt,
            )


            print(
                f"  [LAST] epoch={current_epoch} "
                f"loss={mean_loss:.6f}"
            )


        # ====================================================
        # 2. SAVE best.pth
        # ====================================================
        #
        # CHỈ ghi đè khi loss tốt hơn.
        # ====================================================

        if is_best:

            atomic_torch_save(
                checkpoint_state,

                best_ckpt,
            )


            print(
                f"  [BEST] epoch={best_epoch} "
                f"loss={best_loss:.6f}"
            )


    # ========================================================
    # COMPLETE
    # ========================================================

    metadata = {

        "category":
            category,

        "repo_commit":
            COMMIT,

        "setting":
            "single-class",

        "encoder":
            "dinov2reg_vit_base_14",

        "input_size":
            INPUT_SIZE,

        "crop_size":
            CROP_SIZE,

        "inp_num":
            INP_NUM,

        "epochs":
            EPOCHS,

        "train_batch":
            TRAIN_BATCH,

        "num_workers":
            SAFE_NUM_WORKERS,

        "seed":
            SEED,

        "train_images":
            len(train_data),

        "best_epoch":
            best_epoch,

        "best_train_loss":
            best_loss,

        "best_checkpoint":
            str(best_ckpt),

        "last_checkpoint":
            str(last_ckpt),
    }


    (
        out_dir
        / "metadata.json"
    ).write_text(
        json.dumps(
            metadata,
            indent=2,
        )
    )


    # ========================================================
    # CLEAN
    # ========================================================

    del model
    del trainable
    del optimizer
    del scheduler
    del train_loader

    torch.cuda.empty_cache()


    print(
        "=" * 70
    )

    print(
        "[TRAIN COMPLETE]",
        category,
    )

    print(
        "BEST:",
        best_ckpt,
    )

    print(
        "LAST:",
        last_ckpt,
    )

    print(
        f"Best epoch: {best_epoch}"
    )

    print(
        f"Best loss : {best_loss:.6f}"
    )

    print(
        "=" * 70
    )


    return {
        "best":
            best_ckpt,

        "last":
            last_ckpt,

        "best_epoch":
            best_epoch,

        "best_loss":
            best_loss,
    }


# ============================================================
# RUN
# ============================================================

checkpoint = train_selected_category(
    CATEGORY,

    force=FORCE_TRAIN,
)


print(
    checkpoint
)

[LOAD RESUME] /content/drive/MyDrive/[Q3-4] 2026/[S7] Computer Vision/CV-Nhóm 9/inp_former_results_single_class/saved_results/INP-Former-Single-Class_dataset=MVTec-AD2_Encoder=dinov2reg_vit_base_14_Resize=448_Crop=392_INP_num=6/can/last.pth
[MIGRATE] Created best.pth from existing epoch 26
[RESUME] can: last completed epoch = 26
[BEST] epoch=26, loss=0.226705
TRAIN CATEGORY: can
Train images  : 412
Epochs        : 200
Start epoch   : 27
Batch         : 16
Batches/epoch : 25
Data workers  : 0
GPU           : Tesla T4
Best loss     : 0.226705


can 27/200:   0%|          | 0/25 [00:00<?, ?it/s]

can: epoch 027/200 loss=0.222226
  [LAST] epoch=27 loss=0.222226
  [BEST] epoch=27 loss=0.222226


can 28/200:   0%|          | 0/25 [00:00<?, ?it/s]

can: epoch 028/200 loss=0.221147
  [LAST] epoch=28 loss=0.221147
  [BEST] epoch=28 loss=0.221147


can 29/200:   0%|          | 0/25 [00:00<?, ?it/s]

can: epoch 029/200 loss=0.216943
  [LAST] epoch=29 loss=0.216943


### Không dùng: batch train cả 8

Để tránh vô tình chiếm GPU quá lâu, phần batch được giữ **comment hoàn toàn**:

```python
# for category in ALL_CATEGORIES:
#     train_selected_category(category)
```

Hãy đổi `CATEGORY` ở Cell 1 và chạy từng phiên.

## 7. Public inference — chỉ selected category

In [ ]:
RAW_ROOT = (
    DRIVE_OUT
    / "raw_maps"
)

CROP_ROOT = (
    RAW_ROOT
    / "official_crop"
)

FULL_ROOT = (
    RAW_ROOT
    / "fullres"
)

PRIVATE_TIFF_ROOT = (
    DRIVE_OUT
    / "anomaly_images"
)


# ============================================================
# TẠO THƯ MỤC OUTPUT
# ============================================================
# Không liên quan tới category.
# Chỉ tạo 3 thư mục gốc.

for path in [
    CROP_ROOT,
    FULL_ROOT,
    PRIVATE_TIFF_ROOT,
]:
    path.mkdir(
        parents=True,
        exist_ok=True,
    )


# ============================================================
# GAUSSIAN SMOOTHING
# ============================================================

gaussian_kernel = get_gaussian_kernel(
    kernel_size=5,
    sigma=4,
).to(device)


# ============================================================
# LOAD MODEL CỦA CATEGORY ĐANG CHỌN
# ============================================================

# def load_trained_model(category):

#     ckpt = (
#         CHECKPOINT_ROOT
#         / category
#         / "last.pth"
#     )

#     assert ckpt.is_file(), (
#         "Chưa có checkpoint.\n"
#         f"Hãy train category '{category}' trước:\n"
#         f"{ckpt}"
#     )

#     print(
#         "Loading checkpoint:",
#         ckpt,
#     )

#     model, _ = build_model()

#     model.load_state_dict(
#         torch.load(
#             ckpt,
#             map_location="cpu",
#         ),
#         strict=True,
#     )

#     model.eval()

#     return model
#test tạm
def load_trained_model(category):

    ckpt = (
        CHECKPOINT_ROOT
        / category
        / "last.pth"
    )

    assert ckpt.is_file(), (
        f"Không tìm thấy last.pth:\n{ckpt}"
    )

    print("=" * 70)
    print("LOAD INTERMEDIATE CHECKPOINT")
    print("Category   :", category)
    print("Checkpoint :", ckpt)

    # last.pth do chính notebook này tạo ra,
    # chứa model + optimizer + scheduler.
    state = torch.load(
        ckpt,
        map_location="cpu",
        weights_only=False,
    )

    assert "model" in state
    assert "epoch" in state

    epoch_done = int(
        state["epoch"]
    ) + 1

    print(
        f"Checkpoint epoch: "
        f"{epoch_done}/{EPOCHS}"
    )

    model, _ = build_model()

    model.load_state_dict(
        state["model"],
        strict=True,
    )

    model.eval()

    print(
        f"Loaded {category} "
        f"at epoch {epoch_done} successfully."
    )

    print("=" * 70)

    return model


# ============================================================
# PREDICT ANOMALY MAP
# ============================================================

@torch.inference_mode()
def predict_official_crop_map(
    model,
    image,
):

    image = image.to(
        device,
        non_blocking=True,
    )

    en, de, _ = model(
        image
    )

    anomaly_map, _ = cal_anomaly_maps(
        en,
        de,
        image.shape[-1],
    )

    anomaly_map = gaussian_kernel(
        anomaly_map
    )

    return anomaly_map[:, 0]


# ============================================================
# PUBLIC INFERENCE
# CHỈ CHẠY CATEGORY ĐANG CHỌN
# ============================================================

def export_public_selected(
    category,
    force=False,
):

    print(
        "=" * 70
    )

    print(
        "PUBLIC INFERENCE CATEGORY:",
        category,
    )

    print(
        "=" * 70
    )


    # --------------------------------------------------------
    # DATASET
    # --------------------------------------------------------

    test_data = MVTecDataset(
        root=str(
            INP_DATA_ROOT
            / category
        ),
        transform=data_transform,
        gt_transform=gt_transform,
        phase="test",
    )


    # --------------------------------------------------------
    # DATALOADER
    # --------------------------------------------------------
    # num_workers=0 để tránh lỗi multiprocessing trên Colab.

    test_loader = DataLoader(
        test_data,
        batch_size=1,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
    )


    # --------------------------------------------------------
    # MODEL
    # --------------------------------------------------------

    model = load_trained_model(
        category
    )


    rows = []


    # --------------------------------------------------------
    # INFERENCE
    # --------------------------------------------------------

    for (
        image,
        gt,
        label,
        img_path,
    ) in tqdm(
        test_loader,
        desc=f"PUBLIC {category}",
    ):

        src = Path(
            img_path[0]
        )

        kind = (
            src.parent.name
        )

        stem = (
            src.stem
        )


        # ====================================================
        # OUTPUT PATH
        # ====================================================

        crop_dst = (
            CROP_ROOT
            / category
            / kind
            / f"{stem}.npy"
        )

        full_dst = (
            FULL_ROOT
            / category
            / kind
            / f"{stem}.npy"
        )


        crop_dst.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        full_dst.parent.mkdir(
            parents=True,
            exist_ok=True,
        )


        # ====================================================
        # SKIP NẾU ĐÃ CÓ
        # ====================================================

        if (
            crop_dst.exists()
            and full_dst.exists()
            and not force
        ):

            crop_map = np.load(
                crop_dst
            )


        # ====================================================
        # INFERENCE MỚI
        # ====================================================

        else:

            crop_map = (
                predict_official_crop_map(
                    model,
                    image,
                )[0]
                .float()
                .cpu()
                .numpy()
                .astype(
                    np.float32
                )
            )


            # -----------------------------------------------
            # SAVE OFFICIAL 392x392 MAP
            # -----------------------------------------------

            np.save(
                crop_dst,
                crop_map,
            )


            # -----------------------------------------------
            # ORIGINAL IMAGE SIZE
            # -----------------------------------------------

            with Image.open(
                src
            ) as im:

                width, height = (
                    im.size
                )


            # -----------------------------------------------
            # UPSAMPLE VỀ FULL RESOLUTION
            # -----------------------------------------------

            full_map = cv2.resize(
                crop_map,
                (
                    width,
                    height,
                ),
                interpolation=cv2.INTER_LINEAR,
            ).astype(
                np.float32
            )


            np.save(
                full_dst,
                full_map,
            )


        # ====================================================
        # MANIFEST
        # ====================================================

        rows.append({
            "category":
                category,

            "kind":
                kind,

            "image":
                str(src),

            "official_crop_npy":
                str(crop_dst),

            "fullres_npy":
                str(full_dst),

            "score_min":
                float(
                    crop_map.min()
                ),

            "score_max":
                float(
                    crop_map.max()
                ),
        })


    # ========================================================
    # SAVE MANIFEST
    # ========================================================

    manifest = pd.DataFrame(
        rows
    )


    manifest_path = (
        DRIVE_OUT
        / "manifests"
        / f"{category}_test_public.csv"
    )


    manifest_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    manifest.to_csv(
        manifest_path,
        index=False,
    )


    # ========================================================
    # CLEAN GPU
    # ========================================================

    del model

    torch.cuda.empty_cache()


    print(
        "=" * 70
    )

    print(
        "PUBLIC INFERENCE COMPLETE:",
        category,
    )

    print(
        "Images:",
        len(manifest),
    )

    print(
        "Manifest:",
        manifest_path,
    )

    print(
        "=" * 70
    )


    return manifest


# ============================================================
# CHỈ CHẠY 1 CATEGORY
# ============================================================

public_manifest = export_public_selected(
    CATEGORY,
    force=FORCE_PUBLIC_INFERENCE,
)


display(
    public_manifest.head()
)


print(
    "PUBLIC INFERENCE DONE:",
    CATEGORY,
)

PUBLIC INFERENCE CATEGORY: can
LOAD INTERMEDIATE CHECKPOINT
Category   : can
Checkpoint : /content/drive/MyDrive/[Q3-4] 2026/[S7] Computer Vision/CV-Nhóm 9/inp_former_results_single_class/saved_results/INP-Former-Single-Class_dataset=MVTec-AD2_Encoder=dinov2reg_vit_base_14_Resize=448_Crop=392_INP_num=6/can/last.pth
Checkpoint epoch: 26/200
Loaded can at epoch 26 successfully.


PUBLIC can:   0%|          | 0/162 [00:00<?, ?it/s]

PUBLIC INFERENCE COMPLETE: can
Images: 162
Manifest: /content/drive/MyDrive/[Q3-4] 2026/[S7] Computer Vision/CV-Nhóm 9/inp_former_results_single_class/manifests/can_test_public.csv


,category,kind,image,official_crop_npy,fullres_npy,score_min,score_max
0,can,good,/content/INP-Former/data/mvtec_ad_2_inp/can/te...,/content/drive/MyDrive/[Q3-4] 2026/[S7] Comput...,/content/drive/MyDrive/[Q3-4] 2026/[S7] Comput...,0.024830,0.303772
1,can,good,/content/INP-Former/data/mvtec_ad_2_inp/can/te...,/content/drive/MyDrive/[Q3-4] 2026/[S7] Comput...,/content/drive/MyDrive/[Q3-4] 2026/[S7] Comput...,0.023294,0.307463
2,can,good,/content/INP-Former/data/mvtec_ad_2_inp/can/te...,/content/drive/MyDrive/[Q3-4] 2026/[S7] Comput...,/content/drive/MyDrive/[Q3-4] 2026/[S7] Comput...,0.020551,0.294219
3,can,good,/content/INP-Former/data/mvtec_ad_2_inp/can/te...,/content/drive/MyDrive/[Q3-4] 2026/[S7] Comput...,/content/drive/MyDrive/[Q3-4] 2026/[S7] Comput...,0.024904,0.446844
4,can,good,/content/INP-Former/data/mvtec_ad_2_inp/can/te...,/content/drive/MyDrive/[Q3-4] 2026/[S7] Comput...,/content/drive/MyDrive/[Q3-4] 2026/[S7] Comput...,0.022217,0.298175


PUBLIC INFERENCE DONE: can


## 8. AU-PRO@0.05 — selected category

In [ ]:
MAX_FPR = 0.05

def compute_aupro_005(
    normal_scores,
    region_scores,
):
    normal_scores = np.concatenate(
        normal_scores
    ).astype(
        np.float32,
        copy=False,
    )

    num_regions = len(
        region_scores
    )

    if num_regions == 0:
        raise RuntimeError(
            "Không tìm thấy GT connected component."
        )

    anomaly_scores = []
    anomaly_weights = []

    # Mỗi GT region có trọng số bằng nhau.
    for scores in region_scores:
        scores = np.asarray(
            scores,
            dtype=np.float32,
        ).reshape(-1)

        if scores.size == 0:
            continue

        anomaly_scores.append(
            scores
        )

        anomaly_weights.append(
            np.full(
                scores.size,
                1.0
                / (
                    num_regions
                    * scores.size
                ),
                dtype=np.float64,
            )
        )

    anomaly_scores = np.concatenate(
        anomaly_scores
    )

    anomaly_weights = np.concatenate(
        anomaly_weights
    )

    num_normal_pixels = (
        normal_scores.size
    )

    if num_normal_pixels == 0:
        raise RuntimeError(
            "Không có normal pixels để tính FPR."
        )

    scores_all = np.concatenate([
        normal_scores,
        anomaly_scores,
    ])

    fpr_delta = np.concatenate([
        np.full(
            num_normal_pixels,
            1.0
            / num_normal_pixels,
            dtype=np.float64,
        ),
        np.zeros(
            anomaly_scores.size,
            dtype=np.float64,
        ),
    ])

    pro_delta = np.concatenate([
        np.zeros(
            num_normal_pixels,
            dtype=np.float64,
        ),
        anomaly_weights,
    ])

    order = np.argsort(
        scores_all,
        kind="mergesort",
    )[::-1]

    sorted_scores = (
        scores_all[order]
    )

    cumulative_fpr = np.cumsum(
        fpr_delta[order]
    )

    cumulative_pro = np.cumsum(
        pro_delta[order]
    )

    group_ends = np.r_[
        np.flatnonzero(
            sorted_scores[1:]
            != sorted_scores[:-1]
        ),
        sorted_scores.size - 1,
    ]

    fpr = np.r_[
        0.0,
        cumulative_fpr[group_ends],
    ]

    pro = np.r_[
        0.0,
        cumulative_pro[group_ends],
    ]

    unique_fpr, first_idx, counts = np.unique(
        fpr,
        return_index=True,
        return_counts=True,
    )

    unique_pro = pro[
        first_idx
        + counts
        - 1
    ]

    if unique_fpr[-1] < MAX_FPR:
        raise RuntimeError(
            "FPR curve không đạt 0.05."
        )

    idx = np.searchsorted(
        unique_fpr,
        MAX_FPR,
        side="left",
    )

    if (
        idx < len(unique_fpr)
        and np.isclose(
            unique_fpr[idx],
            MAX_FPR,
        )
    ):
        x = unique_fpr[
            :idx + 1
        ]

        y = unique_pro[
            :idx + 1
        ]

    else:
        x0 = unique_fpr[
            idx - 1
        ]

        x1 = unique_fpr[
            idx
        ]

        y0 = unique_pro[
            idx - 1
        ]

        y1 = unique_pro[
            idx
        ]

        y_at_max_fpr = (
            y0
            + (
                MAX_FPR
                - x0
            )
            * (
                y1
                - y0
            )
            / (
                x1
                - x0
            )
        )

        x = np.r_[
            unique_fpr[:idx],
            MAX_FPR,
        ]

        y = np.r_[
            unique_pro[:idx],
            y_at_max_fpr,
        ]

    area = np.trapezoid(
        y,
        x,
    )

    return float(
        area
        / MAX_FPR
    )

def evaluate_selected_category(
    category,
):
    dataset = MVTecDataset(
        root=str(
            INP_DATA_ROOT
            / category
        ),
        transform=data_transform,
        gt_transform=gt_transform,
        phase="test",
    )

    normal_scores = []
    region_scores = []

    n_good = 0
    n_bad = 0

    for idx in tqdm(
        range(len(dataset)),
        desc=f"AU-PRO0.05 {category}",
    ):
        (
            _,
            gt,
            label,
            img_path,
        ) = dataset[idx]

        src = Path(
            img_path
        )

        kind = (
            src.parent.name
        )

        map_path = (
            CROP_ROOT
            / category
            / kind
            / f"{src.stem}.npy"
        )

        assert map_path.is_file(), (
            "Thiếu public raw map:\n"
            f"{map_path}"
        )

        anomaly_map = np.load(
            map_path,
            mmap_mode="r",
        ).astype(
            np.float32,
            copy=False,
        )

        gt_np = (
            gt[0]
            .numpy()
            > 0.5
        ).astype(
            np.uint8
        )

        assert (
            anomaly_map.shape
            == gt_np.shape
        ), (
            src,
            anomaly_map.shape,
            gt_np.shape,
        )

        normal_scores.append(
            anomaly_map[
                gt_np == 0
            ].reshape(-1)
        )

        if gt_np.any():
            (
                num_labels,
                labels,
            ) = cv2.connectedComponents(
                gt_np,
                connectivity=8,
            )

            for region_id in range(
                1,
                num_labels,
            ):
                region_scores.append(
                    anomaly_map[
                        labels
                        == region_id
                    ].reshape(-1)
                )

        if int(label) == 0:
            n_good += 1
        else:
            n_bad += 1

    score = compute_aupro_005(
        normal_scores,
        region_scores,
    )

    result = {
        "Category":
            category,
        "AU_PRO_0.05":
            score,
        "AU_PRO_0.05_pct":
            score * 100.0,
        "good_images":
            n_good,
        "bad_images":
            n_bad,
        "gt_regions":
            len(region_scores),
    }

    return result

result = evaluate_selected_category(
    CATEGORY
)

result_df = pd.DataFrame([
    result
])

metric_dir = (
    DRIVE_OUT
    / "metrics"
)

metric_dir.mkdir(
    parents=True,
    exist_ok=True,
)

metric_path = (
    metric_dir
    / f"{CATEGORY}.csv"
)

result_df.to_csv(
    metric_path,
    index=False,
)

display(result_df)

print(
    f"{CATEGORY} AU-PRO@0.05 = "
    f"{result['AU_PRO_0.05_pct']:.4f}%"
)

print(
    "Saved:",
    metric_path,
)

AU-PRO0.05 can:   0%|          | 0/162 [00:00<?, ?it/s]

,Category,AU_PRO_0.05,AU_PRO_0.05_pct,good_images,bad_images,gt_regions
0,can,0.05,5.000009,72,90,90


can AU-PRO@0.05 = 5.0000%
Saved: /content/drive/MyDrive/[Q3-4] 2026/[S7] Computer Vision/CV-Nhóm 9/inp_former_results_single_class/metrics/can.csv


## 9. Tổng hợp kết quả đã chạy — KHÔNG dùng GPU

Cell này chỉ đọc các file `metrics/<category>.csv` đã tích lũy trên Drive.

- Chạy 1 category → bảng có 1 dòng.
- Chạy 4 category → bảng có 4 dòng.
- Khi đủ 8 → tự thêm `macro_mean` và ghi `week04_primary.csv`.

Không cần giữ 8 model cùng lúc trên GPU.

In [ ]:
metric_dir = (
    DRIVE_OUT
    / "metrics"
)

available_rows = []
missing_categories = []

for category in ALL_CATEGORIES:
    path = (
        metric_dir
        / f"{category}.csv"
    )

    if path.is_file():
        one = pd.read_csv(path)

        assert len(one) == 1, (
            category,
            path,
        )

        available_rows.append(
            one.iloc[0].to_dict()
        )

    else:
        missing_categories.append(
            category
        )

summary_df = pd.DataFrame(
    available_rows
)

if not summary_df.empty:
    order_map = {
        category: idx
        for idx, category
        in enumerate(
            ALL_CATEGORIES
        )
    }

    summary_df["_order"] = (
        summary_df["Category"]
        .map(order_map)
    )

    summary_df = (
        summary_df
        .sort_values("_order")
        .drop(columns="_order")
        .reset_index(drop=True)
    )

display(summary_df)

print(
    "Completed:",
    summary_df["Category"].tolist()
    if not summary_df.empty
    else []
)

print(
    "Missing  :",
    missing_categories,
)

if len(missing_categories) == 0:
    final_df = summary_df.copy()

    macro_row = {
        "Category":
            "macro_mean",
        "AU_PRO_0.05":
            final_df[
                "AU_PRO_0.05"
            ].mean(),
        "AU_PRO_0.05_pct":
            final_df[
                "AU_PRO_0.05_pct"
            ].mean(),
        "good_images":
            final_df[
                "good_images"
            ].sum(),
        "bad_images":
            final_df[
                "bad_images"
            ].sum(),
        "gt_regions":
            final_df[
                "gt_regions"
            ].sum(),
    }

    final_df.loc[
        len(final_df)
    ] = macro_row

    primary_path = (
        DRIVE_OUT
        / "week04_primary.csv"
    )

    final_df.to_csv(
        primary_path,
        index=False,
    )

    print(
        "\n8/8 COMPLETE ->",
        primary_path,
    )

    display(final_df)

else:
    partial_path = (
        DRIVE_OUT
        / "week04_primary_partial.csv"
    )

    summary_df.to_csv(
        partial_path,
        index=False,
    )

    print(
        "\nChưa đủ 8 category."
    )

    print(
        "Partial table:",
        partial_path,
    )

## 10. OPTIONAL — private / private-mixed inference

Mặc định:
```python
RUN_PRIVATE_INFERENCE = False
```

Chỉ bật `True` ở Cell 1 sau khi public AU-PRO của selected category đã hợp lệ.

Cell này cũng chỉ inference **selected category**, không chạy 8 category.

In [ ]:
class PrivateMAD2Dataset(
    Dataset
):
    def __init__(
        self,
        root,
        category,
        split,
        transform,
    ):
        self.paths = sorted(
            (
                Path(root)
                / category
                / split
            ).glob("*.png")
        )

        assert self.paths, (
            f"Không có ảnh: "
            f"{root}/{category}/{split}"
        )

        self.transform = transform

    def __len__(self):
        return len(
            self.paths
        )

    def __getitem__(
        self,
        idx,
    ):
        path = self.paths[
            idx
        ]

        with Image.open(path) as im:
            rgb = im.convert(
                "RGB"
            )

            width, height = (
                rgb.size
            )

            image = self.transform(
                rgb
            )

        return (
            image,
            str(path),
            height,
            width,
        )

def export_private_selected(
    category,
    split,
):
    dataset = PrivateMAD2Dataset(
        MAD2_NATIVE_ROOT,
        category,
        split,
        data_transform,
    )

    loader = DataLoader(
        dataset,
        batch_size=1,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    model = load_trained_model(
        category
    )

    for (
        image,
        image_path,
        height,
        width,
    ) in tqdm(
        loader,
        desc=(
            f"{category}/"
            f"{split}"
        ),
    ):
        src = Path(
            image_path[0]
        )

        height = int(
            height[0]
        )

        width = int(
            width[0]
        )

        crop_dst = (
            CROP_ROOT
            / category
            / split
            / f"{src.stem}.npy"
        )

        full_dst = (
            FULL_ROOT
            / category
            / split
            / f"{src.stem}.npy"
        )

        tiff_dst = (
            PRIVATE_TIFF_ROOT
            / category
            / split
            / f"{src.stem}.tiff"
        )

        for parent in [
            crop_dst.parent,
            full_dst.parent,
            tiff_dst.parent,
        ]:
            parent.mkdir(
                parents=True,
                exist_ok=True,
            )

        if (
            crop_dst.exists()
            and full_dst.exists()
            and tiff_dst.exists()
        ):
            continue

        crop_map = (
            predict_official_crop_map(
                model,
                image,
            )[0]
            .float()
            .cpu()
            .numpy()
            .astype(np.float32)
        )

        np.save(
            crop_dst,
            crop_map,
        )

        full_map = cv2.resize(
            crop_map,
            (width, height),
            interpolation=cv2.INTER_LINEAR,
        ).astype(np.float32)

        np.save(
            full_dst,
            full_map,
        )

        tifffile.imwrite(
            tiff_dst,
            full_map.astype(
                np.float16
            ),
        )

    del model

    torch.cuda.empty_cache()

if RUN_PRIVATE_INFERENCE:
    PRIVATE_SPLITS = [
        "test_private",
        "test_private_mixed",
    ]

    for split in PRIVATE_SPLITS:
        export_private_selected(
            CATEGORY,
            split,
        )

    print(
        "PRIVATE DONE:",
        CATEGORY,
    )

else:
    print(
        "SKIP PRIVATE. "
        "Set RUN_PRIVATE_INFERENCE=True "
        "ở Cell 1 nếu cần."
    )

## 11. OPTIONAL — failure gallery selected category

Mặc định tắt để không làm thêm việc khi train.

Bật:
```python
RUN_FAILURE_ANALYSIS = True
```

Cell chỉ xử lý category đang chọn.

In [ ]:
if RUN_FAILURE_ANALYSIS:
    import matplotlib.pyplot as plt

    gallery_root = (
        DRIVE_OUT
        / "failure_gallery"
        / CATEGORY
    )

    gallery_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    bad_paths = sorted(
        (
            MAD2_NATIVE_ROOT
            / CATEGORY
            / "test_public"
            / "bad"
        ).glob("*.png")
    )

    for image_path in bad_paths:
        full_map_path = (
            FULL_ROOT
            / CATEGORY
            / "bad"
            / f"{image_path.stem}.npy"
        )

        if not full_map_path.is_file():
            continue

        gt_path = (
            MAD2_NATIVE_ROOT
            / CATEGORY
            / "test_public"
            / "ground_truth"
            / "bad"
            / f"{image_path.stem}_mask.png"
        )

        image = np.array(
            Image.open(
                image_path
            ).convert("RGB")
        )

        gt = (
            np.array(
                Image.open(
                    gt_path
                )
            )
            > 0
        )

        anomaly_map = np.load(
            full_map_path
        )

        fig = plt.figure(
            figsize=(18, 5)
        )

        ax = fig.add_subplot(
            1, 3, 1
        )
        ax.imshow(image)
        ax.set_title("Input")
        ax.axis("off")

        ax = fig.add_subplot(
            1, 3, 2
        )
        ax.imshow(
            gt,
            cmap="gray",
        )
        ax.set_title("GT")
        ax.axis("off")

        ax = fig.add_subplot(
            1, 3, 3
        )
        ax.imshow(image)
        ax.imshow(
            anomaly_map,
            cmap="jet",
            alpha=0.45,
        )
        ax.set_title(
            "INP-Former"
        )
        ax.axis("off")

        fig.tight_layout()

        fig.savefig(
            gallery_root
            / f"{image_path.stem}.png",
            dpi=150,
        )

        plt.close(fig)

    print(
        "Gallery:",
        gallery_root,
    )

else:
    print(
        "SKIP FAILURE ANALYSIS. "
        "Set RUN_FAILURE_ANALYSIS=True "
        "ở Cell 1 nếu cần."
    )

## 12. Audit selected category

In [ ]:
category_ckpt = (
    CHECKPOINT_ROOT
    / CATEGORY
    / "model.pth"
)

category_metric = (
    DRIVE_OUT
    / "metrics"
    / f"{CATEGORY}.csv"
)

public_good_maps = list(
    (
        CROP_ROOT
        / CATEGORY
        / "good"
    ).glob("*.npy")
)

public_bad_maps = list(
    (
        CROP_ROOT
        / CATEGORY
        / "bad"
    ).glob("*.npy")
)

expected_good = len(
    list(
        (
            MAD2_NATIVE_ROOT
            / CATEGORY
            / "test_public"
            / "good"
        ).glob("*.png")
    )
)

expected_bad = len(
    list(
        (
            MAD2_NATIVE_ROOT
            / CATEGORY
            / "test_public"
            / "bad"
        ).glob("*.png")
    )
)

audit = {
    "category":
        CATEGORY,

    "checkpoint_exists":
        category_ckpt.is_file(),

    "checkpoint_sha256":
        (
            sha256_file(
                category_ckpt
            )
            if category_ckpt.is_file()
            else None
        ),

    "metric_exists":
        category_metric.is_file(),

    "public_good_expected":
        expected_good,

    "public_good_maps":
        len(public_good_maps),

    "public_bad_expected":
        expected_bad,

    "public_bad_maps":
        len(public_bad_maps),

    "repo_commit":
        COMMIT,

    "setting":
        "single-class",

    "preprocess":
        (
            "Resize 448 -> "
            "CenterCrop 392 -> "
            "ImageNet Normalize"
        ),
}

audit_dir = (
    DRIVE_OUT
    / "audit"
)

audit_dir.mkdir(
    parents=True,
    exist_ok=True,
)

audit_path = (
    audit_dir
    / f"{CATEGORY}.json"
)

audit_path.write_text(
    json.dumps(
        audit,
        indent=2,
    )
)

print(
    json.dumps(
        audit,
        indent=2,
    )
)

print(
    "Audit saved:",
    audit_path,
)

## Workflow cho 8 category

Không train batch. Lặp lại quy trình này **8 phiên độc lập**:

```text
CATEGORY = "can"
Run all
    ↓
CATEGORY = "fabric"
Restart runtime + Run all
    ↓
CATEGORY = "fruit_jelly"
...
```

Danh sách:

```python
# CATEGORY = "can"
# CATEGORY = "fabric"
# CATEGORY = "fruit_jelly"
# CATEGORY = "rice"
# CATEGORY = "sheet_metal"
# CATEGORY = "vial"
# CATEGORY = "wallplugs"
# CATEGORY = "walnuts"
```

Mỗi phiên chỉ giữ **1 model trên GPU**. Kết quả được tích lũy trên Drive. Khi đã có đủ `metrics/*.csv`, Cell 9 tự sinh:

```text
week04_primary.csv
```

với 8 AU-PRO@0.05 per-category + `macro_mean`.